# Dijet event display (Plotly)

Picks a single Pythia8 QCD dijet event, clusters it with FastJet anti-$k_t$
(same setup as [demo_pythia_fastjet.ipynb](demo_pythia_fastjet.ipynb)), and
renders it as an interactive Plotly event display: an $\eta$-$\phi$ jet-cone
map, a 3D $p_T$ "lego" view, and a cylindrical view with particles
emerging from the collision point.

In [ ]:
# heppyyier.load() is a no-op if packages were already loaded via
# `module load` or the heppyyier Jupyter kernel. Safe to leave in place.
import heppyyier
heppyyier.load('pythia8')
heppyyier.load('fastjet')

In [ ]:
import cppyy
import pythia8
import fastjet
import numpy as np

try:
    import plotly.graph_objects as go
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'plotly'], check=True)
    import plotly.graph_objects as go

# Sanity check: cppyy location
print(f"cppyy from: {cppyy.__file__}")

PseudoJetVec = cppyy.gbl.std.vector[fastjet.PseudoJet]

def wrap_phi(phi):
    """Map any phi convention onto the standard (-pi, pi] range.

    FastJet's PseudoJet.phi() returns [0, 2*pi); Pythia8's Particle.phi()
    already returns (-pi, pi]. Route everything through this before
    plotting so jet axes/cones line up with their constituents.
    """
    return (phi + np.pi) % (2 * np.pi) - np.pi

## Configure Pythia8

In [ ]:
pythia = pythia8.Pythia()
pythia.readString('Beams:eCM = 13000.')
pythia.readString('HardQCD:all = on')
pythia.readString('PhaseSpace:pTHatMin = 20.')
pythia.readString('Next:numberShowEvent = 0')
pythia.readString('Print:quiet = on')
pythia.init()

## Jet definition

In [ ]:
R      = 0.4
pt_min = 20.0

jet_def = fastjet.JetDefinition(fastjet.antikt_algorithm, R)

## Select a dijet event

Generate events until a clean two-jet topology turns up: at least two jets
above `pt_min`, with the subleading jet carrying a healthy fraction of the
leading jet's $p_T$ (an actual back-to-back pair, not one hard jet plus soft
radiation). Each final-state particle gets tagged with its position via
`set_user_index` so constituents can be traced back to it after clustering.

In [ ]:
def get_dijet_event(max_tries=500, min_pt_ratio=0.3):
    """Generate events until a clean dijet turns up.

    Returns (cs, jets, parts) where `parts` is a list of dicts with the
    kinematics of every final, visible particle in the selected event, in
    the same order as their PseudoJet user_index.
    """
    for _ in range(max_tries):
        if not pythia.next():
            continue

        parts = []
        particles = PseudoJetVec()
        for i in range(pythia.event.size()):
            p = pythia.event[i]
            if p.isFinal() and p.isVisible():
                pj = fastjet.PseudoJet(p.px(), p.py(), p.pz(), p.e())
                pj.set_user_index(len(parts))
                particles.push_back(pj)
                parts.append({'pt': p.pT(), 'eta': p.eta(), 'phi': p.phi(),
                              'px': p.px(), 'py': p.py(), 'pz': p.pz(),
                              'id': p.id(), 'name': p.name()})

        cs   = fastjet.ClusterSequence(particles, jet_def)
        jets = fastjet.sorted_by_pt(cs.inclusive_jets(pt_min))

        if len(jets) >= 2 and jets[1].pt() >= min_pt_ratio * jets[0].pt():
            return cs, jets, parts

    raise RuntimeError(f'no clean dijet event found in {max_tries} tries')

cs, jets, parts = get_dijet_event()

print(f'{len(parts)} final-state visible particles -> {len(jets)} jets\n')
print(f"  {'#':<4} {'pT':>8} {'eta':>8} {'phi':>8} {'mass':>8} {'n_const':>8}")
print('  ' + '-' * 52)
for j, jet in enumerate(jets):
    nc = len(cs.constituents(jet))
    print(f'  {j:<4} {jet.pt():>8.2f} {jet.eta():>8.3f} {wrap_phi(jet.phi()):>8.3f} {jet.m():>8.3f} {nc:>8}')

## Map particles to jets

Tag every particle with the index of the jet it ended up in (`-1` for
particles left out of both leading jets — underlying event / soft
radiation). All phi values are routed through `wrap_phi` so everything
lives on the standard $(-\pi, \pi]$ range.

In [ ]:
jet_index = [-1] * len(parts)
for j, jet in enumerate(jets):
    for c in cs.constituents(jet):
        jet_index[c.user_index()] = j

pt  = np.array([p['pt']  for p in parts])
eta = np.array([p['eta'] for p in parts])
phi = np.array([wrap_phi(p['phi']) for p in parts])
px  = np.array([p['px']  for p in parts])
py  = np.array([p['py']  for p in parts])
pz  = np.array([p['pz']  for p in parts])
jix = np.array(jet_index)

jet_phi = [wrap_phi(jet.phi()) for jet in jets]

## $\eta$-$\phi$ event display

In [ ]:
PALETTE = ['#4C78A8', '#E45756', '#54A24B', '#F58518', '#B279A2', '#9D755D']

def jet_circle(jet, jphi, radius=R, n=64):
    """Points tracing the jet's clustering-radius cone in the eta-phi plane."""
    t = np.linspace(0, 2 * np.pi, n)
    return jet.eta() + radius * np.cos(t), jphi + radius * np.sin(t)

fig = go.Figure()

# Underlying event / soft radiation not assigned to either leading jet
mask = jix < 0
fig.add_trace(go.Scatter(
    x=eta[mask], y=phi[mask], mode='markers', name='other particles',
    marker=dict(size=np.clip(pt[mask] * 3, 3, 14), color='lightgray',
                line=dict(width=0)),
    text=[f"pT={p:.2f} GeV" for p in pt[mask]], hoverinfo='text+name'
))

# Jet constituents, colored per jet, sized by particle pT
for j, jet in enumerate(jets):
    m = jix == j
    color = PALETTE[j % len(PALETTE)]
    fig.add_trace(go.Scatter(
        x=eta[m], y=phi[m], mode='markers', name=f'jet {j} constituents',
        marker=dict(size=np.clip(pt[m] * 4, 4, 26), color=color,
                    line=dict(width=0.5, color='white')),
        text=[f"pT={p:.2f} GeV" for p in pt[m]], hoverinfo='text+name'
    ))

    cx, cy = jet_circle(jet, jet_phi[j])
    fig.add_trace(go.Scatter(
        x=cx, y=cy, mode='lines', name=f'jet {j} cone (R={R})',
        line=dict(color=color, dash='dash', width=1.5), hoverinfo='skip'
    ))

    fig.add_trace(go.Scatter(
        x=[jet.eta()], y=[jet_phi[j]], mode='markers', showlegend=False,
        marker=dict(size=14, color=color, symbol='x', line=dict(width=2, color='black')),
        text=[f"jet {j}: pT={jet.pt():.1f} GeV, m={jet.m():.1f} GeV"], hoverinfo='text'
    ))

fig.update_layout(
    title=f'Dijet event -- {len(jets)} jets, R={R}, pT_min={pt_min} GeV, '
          f'leading pT={jets[0].pt():.1f} GeV',
    xaxis_title='eta', yaxis_title='phi [rad]',
    xaxis=dict(range=[-5, 5]),
    yaxis=dict(range=[-np.pi, np.pi],
               tickvals=[-np.pi, -np.pi / 2, 0, np.pi / 2, np.pi],
               ticktext=['-π', '-π/2', '0', 'π/2', 'π']),
    width=800, height=600, template='plotly_white'
)
fig.show()

## 3D $p_T$ lego view

In [ ]:
fig3d = go.Figure()

def add_stems(fig, m, name, color, size):
    xs, ys, zs = eta[m], phi[m], pt[m]
    lx, ly, lz = [], [], []
    for x, y, z in zip(xs, ys, zs):
        lx += [x, x, None]
        ly += [y, y, None]
        lz += [0, z, None]
    fig.add_trace(go.Scatter3d(x=lx, y=ly, z=lz, mode='lines',
                                line=dict(color=color, width=4),
                                name=name, showlegend=False, hoverinfo='skip'))
    fig.add_trace(go.Scatter3d(
        x=xs, y=ys, z=zs, mode='markers', name=name,
        marker=dict(size=size, color=color),
        text=[f"pT={p:.2f} GeV" for p in zs], hoverinfo='text+name'
    ))

add_stems(fig3d, jix < 0, 'other particles', 'lightgray', 2)
for j in range(len(jets)):
    add_stems(fig3d, jix == j, f'jet {j}', PALETTE[j % len(PALETTE)], 3)

fig3d.update_layout(
    title='Dijet event -- pT lego view',
    scene=dict(xaxis_title='eta', yaxis_title='phi [rad]', zaxis_title='pT [GeV]',
               xaxis=dict(range=[-5, 5]), yaxis=dict(range=[-np.pi, np.pi])),
    width=800, height=650, template='plotly_white'
)
fig3d.show()

## Cylindrical view — particles emerging from the collision point

Every final-state particle drawn as a ray from the origin $(0,0,0)$, colored
by the jet it belongs to. Ray *direction* is the particle's true momentum
direction $(p_x, p_y, p_z)/|p|$; ray *length* uses $\log(1+|p|)$ rather than
raw GeV, so soft constituents stay visible next to a several-hundred-GeV jet
instead of collapsing to a point at the origin. The scene box is a cube with
equal axis ranges, so the back-to-back dijet topology reads clearly without
one axis dwarfing the others.

In [ ]:
fig_cyl = go.Figure()

# Direction preserved, radius compressed to log(1+|p|) so low-momentum
# constituents don't collapse to the origin next to a hard jet.
p_mag      = np.sqrt(px**2 + py**2 + pz**2)
p_mag_safe = np.where(p_mag > 0, p_mag, 1.0)
r_scale    = np.log1p(p_mag) / p_mag_safe
rx, ry, rz = px * r_scale, py * r_scale, pz * r_scale

def add_rays(fig, m, name, color, width=2.5, showlegend=True):
    xs, ys, zs = rx[m], ry[m], rz[m]
    lx, ly, lz = [], [], []
    for x, y, z in zip(xs, ys, zs):
        lx += [0, x, None]
        ly += [0, y, None]
        lz += [0, z, None]
    fig.add_trace(go.Scatter3d(x=lx, y=ly, z=lz, mode='lines',
                                line=dict(color=color, width=width),
                                name=name, showlegend=showlegend, hoverinfo='skip'))
    fig.add_trace(go.Scatter3d(
        x=xs, y=ys, z=zs, mode='markers', name=name, showlegend=False,
        marker=dict(size=np.clip(pt[m] * 0.6, 3, 12), color=color),
        text=[f"pT={p:.2f} GeV" for p in pt[m]], hoverinfo='text+name'
    ))

add_rays(fig_cyl, jix < 0, 'other particles', 'lightgray', width=1.5)
for j in range(len(jets)):
    add_rays(fig_cyl, jix == j, f'jet {j}', PALETTE[j % len(PALETTE)], width=2.5)

lim = np.log1p(p_mag).max() * 1.05 if len(p_mag) else 1.0
fig_cyl.update_layout(
    title='Dijet event -- particles emerging from the collision point (radius = log(1+|p|))',
    scene=dict(xaxis_title='px', yaxis_title='py', zaxis_title='pz',
               xaxis=dict(range=[-lim, lim]),
               yaxis=dict(range=[-lim, lim]),
               zaxis=dict(range=[-lim, lim]),
               aspectmode='cube'),
    width=800, height=650, template='plotly_white'
)
fig_cyl.show()

---
Re-run the *Select a dijet event* cell above to sample a different event
(the cells below it will pick up the new selection when re-run).